# The app on AWS Lambda

Follows Part 1 of the course's `week3day2_lab2.ipynb`, on this project's deployment (`docs/TECH_STACK.md`, Deployment): the backend and the dashboard go to Lambda as one zip, behind a public address (a Function URL), with the database on Supabase.

Steps, in order. Each one can be run again:

1. **Names**: everything is called `undercover...` and tagged `project=undercover`.
2. **Check `.env`**: the four settings the app needs (only their names are printed, never their values).
3. **Dashboard**: `npm run build` writes `dashboard/dist`.
4. **Package**: the Linux dependencies and our own code, as one zip.
5. **Log group and role**: the identity the function runs as.
6. **Function**: created the first time, updated in place after that.
7. **Public address** (a Function URL).
8. **Test**: the API, the dashboard and the API key.
9. **Update**: how to publish a change.
10. **Gateway** (Part 2 of the lab): the app's routes as tools for the claim agent, with the app's API key kept in AgentCore Identity.
11. **Policy** (Part 3): a rule on the Gateway that refuses a payment over $1,000.
12. **Teardown**: delete what this notebook made. The AgentCore pieces bill by the month, so delete them when you stop.

In [19]:
from pathlib import Path


def load_env(name):
    """Reads one variable from the first .env found, going up from the current folder."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        env_file = folder / ".env"
        if env_file.exists():
            for line in env_file.read_text(encoding="utf-8").splitlines():
                key, sep, value = line.partition("=")
                if sep and key.strip() == name:
                    return value.strip().strip("\"'")
    raise KeyError(f"{name} not found in any .env")


In [20]:
import pandas as pd
import boto3
from botocore.exceptions import BotoCoreError, ClientError


def load_env_first(names, default=None):
    """The value of the first of these variables found in .env. Without a default, none found is an error."""
    for name in names:
        try:
            return load_env(name)
        except KeyError:
            continue
    if default is None:
        raise KeyError(f"none of {', '.join(names)} found in any .env")
    return default


def make_aws_session():
    """boto3 session with the AWS credentials from .env. Accepts AWS_ACCESS_KEY_ID or AWS_ACCESS_KEY, AWS_SECRET_ACCESS_KEY or AWS_SECRET_KEY, and AWS_SESSION_TOKEN when the keys are temporary. The region is AWS_DEFAULT_REGION or AWS_REGION, and us-east-1 when neither is set."""
    return boto3.Session(
        aws_access_key_id=load_env_first(["AWS_ACCESS_KEY_ID", "AWS_ACCESS_KEY"]),
        aws_secret_access_key=load_env_first(["AWS_SECRET_ACCESS_KEY", "AWS_SECRET_KEY"]),
        aws_session_token=load_env_first(["AWS_SESSION_TOKEN"], default="") or None,  # only temporary credentials (SSO, assumed roles) have one
        region_name=load_env_first(["AWS_DEFAULT_REGION", "AWS_REGION"], default="us-east-1"),
    )


def test_aws(session):
    """Asks AWS who these credentials belong to: one call that needs no permission, so it only fails when the keys or the region are wrong."""
    try:
        identity = session.client("sts").get_caller_identity()
        print(f"OK [{session.region_name}]: AWS credentials accepted")
        print(f"account: {identity['Account']} | identity: {identity['Arn']}")
    except ClientError as e:
        error = e.response["Error"]
        print(f"ERROR {error['Code']}: {error['Message']}")
    except BotoCoreError as e:
        print(f"Connection ERROR: {e}")


aws_session = make_aws_session()
test_aws(aws_session)

OK [us-east-1]: AWS credentials accepted
account: 123456789012 | identity: arn:aws:iam::123456789012:user/undercover


## 1. Names

In [21]:
from pathlib import Path

# Everything we create is named undercover... and tagged project=undercover, so the teardown can find it by name.
FUNCTION_NAME = "undercover-app"
APP_ROLE_NAME = "undercover-app-role"
LOG_GROUP = f"/aws/lambda/{FUNCTION_NAME}"
PROJECT_TAG = {"project": "undercover"}

# The AgentCore pieces (sections 10 and 11)
KEY_PROVIDER_NAME = "undercover-app-key"
GATEWAY_ROLE_NAME = "undercover-gateway-role"
GATEWAY_NAME = "undercover-gateway"
TARGET_NAME = "undercover-app"
ENGINE_NAME = "undercover_policy_engine"

# This notebook is in labs/; the project is one folder up
REPO_DIR = Path.cwd().parent if Path.cwd().name == "labs" else Path.cwd()
assert (REPO_DIR / "backend" / "app").is_dir(), f"{REPO_DIR} is not the project folder"
print(f"project: {REPO_DIR}")

project: c:\Users\alexg\Documents\Projects\udemy\FDE\undercover_mex


## 2. Check `.env`

The deployed app reads the same Supabase database as your laptop, and needs the agent's API key. Three of the four are used by the function itself; `VITE_SUPABASE_PUBLISHABLE_KEY` is built into the dashboard.

In [22]:
# What the function needs. The values stay in .env: this cell prints only whether each one is there and looks right.
NEEDED = ["DATABASE_URL", "VITE_SUPABASE_URL", "VITE_SUPABASE_PUBLISHABLE_KEY", "UNDERCOVER_API_KEY"]
values = {}
for name in NEEDED:
    try:
        values[name] = load_env(name)
    except KeyError:
        values[name] = ""
    print(f"{name}: {'found' if values[name] else 'MISSING: add it to .env'}")

# The two things that go wrong most often
if values["UNDERCOVER_API_KEY"] and len(values["UNDERCOVER_API_KEY"]) < 32:
    print("UNDERCOVER_API_KEY is shorter than 32 characters: the app refuses it. Make one with: python -c \"import secrets; print(secrets.token_urlsafe(32))\"")
if values["DATABASE_URL"] and ":5432/" not in values["DATABASE_URL"]:
    print("DATABASE_URL has no :5432/ . It should be Supabase's session pooler address: the function uses the same one on port 6543 (the transaction pooler).")
assert all(values.values()), "fix the .env first"

DATABASE_URL: found
VITE_SUPABASE_URL: found
VITE_SUPABASE_PUBLISHABLE_KEY: found
UNDERCOVER_API_KEY: found


## 3. The dashboard

`npm run build` builds the dashboard into `dashboard/dist`. The app serves these files beside the API, so the dashboard and the API share one address.

In [23]:
import subprocess
import time

DIST = REPO_DIR / "dashboard" / "dist"


def build_dashboard():
    """npm run build, from the project folder: writes dashboard/dist with the Supabase address and publishable key from .env built in."""
    import shutil

    npm = shutil.which("npm")
    if not npm:
        raise RuntimeError("npm was not found: install Node.js")
    subprocess.run([npm, "run", "build"], cwd=REPO_DIR, check=True)


def check_dashboard():
    """The dashboard build is there, and how old it is (build it again after changing the frontend)."""
    index = DIST / "index.html"
    assert index.is_file(), "no dashboard/dist/index.html: run build_dashboard() or `npm run build` in the project folder"
    minutes = (time.time() - index.stat().st_mtime) / 60
    print(f"dashboard/dist is built ({minutes:.0f} minutes ago, {sum(f.stat().st_size for f in DIST.rglob('*') if f.is_file()) / 1e6:.1f} MB)")


# build_dashboard()   # only after a change to the frontend; `npm run build` in a terminal does the same
check_dashboard()

dashboard/dist is built (42 minutes ago, 0.9 MB)


## 4. The package

Lambda runs our code from a zip file. The packages the backend needs on a server are the `server` group in `backend/pyproject.toml`; `uv export` writes them out at the exact versions in `uv.lock`, and `uv pip install` fetches the **Linux** builds. Then our own code: `backend/`, `data/` and the dashboard's build.

In [24]:
import shutil
import subprocess
import zipfile

BUILD = REPO_DIR / "build"  # not committed
PACKAGE = BUILD / "lambda"
ZIP = BUILD / "undercover-app.zip"
LIMIT_MB = 50  # what Lambda takes in a direct upload


def build_package():
    """Makes build/undercover-app.zip: the Linux dependencies of the `server` group, then backend/, data/ and dashboard/dist."""
    uv = shutil.which("uv")
    if not uv:
        raise RuntimeError("uv was not found (https://docs.astral.sh/uv/)")
    check_dashboard()
    shutil.rmtree(PACKAGE, ignore_errors=True)
    PACKAGE.mkdir(parents=True)
    requirements = BUILD / "requirements.txt"

    # The versions of backend/uv.lock, for the packages the server imports and nothing of the agent's
    subprocess.run([uv, "export", "--project", "backend", "--only-group", "server", "--no-hashes", "--no-emit-project", "--quiet", "--output-file", str(requirements)], cwd=REPO_DIR, check=True)
    # Lambda runs Linux and this computer does not: fetch the Linux builds of those packages
    subprocess.run([uv, "pip", "install", "--quiet", "--target", str(PACKAGE), "--python-platform", "x86_64-manylinux2014", "--python-version", "3.12", "--only-binary", ":all:", "-r", str(requirements)], check=True)

    # Our code. backend/ also holds .venv (hundreds of MB), the tests and caches: none of those goes in
    skip = shutil.ignore_patterns("tests", "__pycache__", ".venv", "*.egg-info", ".pytest_cache", "*.pyc")
    for folder in ("backend", "data"):
        shutil.copytree(REPO_DIR / folder, PACKAGE / folder, ignore=skip)
    shutil.copytree(DIST, PACKAGE / "dashboard" / "dist")

    ZIP.unlink(missing_ok=True)
    shutil.make_archive(str(ZIP.with_suffix("")), "zip", PACKAGE)

    # Three checks that catch the usual mistakes before anything is uploaded
    with zipfile.ZipFile(ZIP) as archive:
        names = archive.namelist()
    for needed in ("backend/main.py", "backend/app/main.py", "data/company.json", "dashboard/dist/index.html"):
        assert needed in names, f"{needed} is not in the zip"
    assert any(name.endswith("-x86_64-linux-gnu.so") for name in names), "the dependencies are not the Linux builds"
    assert not any(".venv" in name for name in names), "the virtual environment got into the zip"
    megabytes = ZIP.stat().st_size / 1e6
    print(f"{ZIP.name}: {megabytes:.1f} MB zipped, {len(names)} files (Lambda takes up to {LIMIT_MB} MB in a direct upload)")
    assert megabytes < LIMIT_MB, "the zip is too big for a direct upload"
    return ZIP


package_zip = build_package()

dashboard/dist is built (42 minutes ago, 0.9 MB)
undercover-app.zip: 21.6 MB zipped, 2778 files (Lambda takes up to 50 MB in a direct upload)


## 5. The log group and the role

The log group is ours, with a retention period, so Lambda does not make one that keeps logs forever. The role is the identity the function runs as: it may write to that log group, and nothing else. This is the first cell that creates things on AWS.

In [25]:
import json
import time
import urllib.error
import urllib.request

from botocore.exceptions import ClientError

logs = aws_session.client("logs")
iam = aws_session.client("iam")
lambda_client = aws_session.client("lambda")
region = aws_session.region_name
account = aws_session.client("sts").get_caller_identity()["Account"]
print(f"working in account {account}, region {region}")

# A public Function URL needs a permission that older boto3 versions do not know: stop now, before anything is created
members = lambda_client.meta.service_model.operation_model("AddPermission").input_shape.members
if "InvokedViaFunctionUrl" not in members:
    raise RuntimeError("this boto3 is too old for a public Function URL: run `%pip install -U boto3` in a cell and restart the kernel")

# The log group is ours, with a retention period, so Lambda does not make one that keeps logs forever
try:
    logs.create_log_group(logGroupName=LOG_GROUP)
    print(f"created log group {LOG_GROUP}")
except logs.exceptions.ResourceAlreadyExistsException:
    print(f"log group {LOG_GROUP} already exists")
logs.put_retention_policy(logGroupName=LOG_GROUP, retentionInDays=7)
print("logs are kept for 7 days")

# The role: Lambda may run the app as it, and the app may write to its log group and do nothing else
lambda_trust = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}],
}
try:
    app_role = iam.create_role(
        RoleName=APP_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(lambda_trust),
        Description="The role the UnderCover app runs as on Lambda",
        Tags=[{"Key": key, "Value": value} for key, value in PROJECT_TAG.items()],
    )["Role"]
    print(f"created role {APP_ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    app_role = iam.get_role(RoleName=APP_ROLE_NAME)["Role"]
    print(f"role {APP_ROLE_NAME} already exists")

logging_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Action": ["logs:CreateLogStream", "logs:PutLogEvents"], "Resource": f"arn:aws:logs:{region}:{account}:log-group:{LOG_GROUP}:*"}],
}
iam.put_role_policy(RoleName=APP_ROLE_NAME, PolicyName="undercover-app-logging", PolicyDocument=json.dumps(logging_policy))
print("the role can write to the log group")

working in account 123456789012, region us-east-1
log group /aws/lambda/undercover-app already exists
logs are kept for 7 days
role undercover-app-role already exists
the role can write to the log group


## 6. The function

First run: creates the function from the zip (a few seconds for a new role to reach Lambda). Later runs: replaces its code and settings in place, so the address stays the same. The settings are `DATABASE_URL`, `VITE_SUPABASE_URL` and `UNDERCOVER_API_KEY`, taken from `.env`.

In [26]:
def lambda_settings():
    """The function's settings. The database address is the Supabase one from .env, on port 6543 (the transaction pooler): on Lambda every request opens its own connection, and the session pooler's 15 would soon run out."""
    database_url = load_env("DATABASE_URL").replace(":5432/", ":6543/")
    assert ":6543/" in database_url, "DATABASE_URL should be Supabase's pooler address, with :5432/ or :6543/ in it"
    return {
        "Runtime": "python3.12",
        "Handler": "backend.main.handler",  # backend/main.py, the object `handler`
        "Timeout": 30,
        "MemorySize": 1024,
        "Environment": {"Variables": {
            "DATABASE_URL": database_url,
            "VITE_SUPABASE_URL": load_env("VITE_SUPABASE_URL"),
            "UNDERCOVER_API_KEY": load_env("UNDERCOVER_API_KEY"),
        }},
    }


def create_function(zip_bytes, settings):
    """Creates the function. A new role can take a few seconds to reach Lambda, so it tries again."""
    for attempt in range(10):
        try:
            return lambda_client.create_function(FunctionName=FUNCTION_NAME, Role=app_role["Arn"], Code={"ZipFile": zip_bytes}, Tags=PROJECT_TAG, **settings)
        except lambda_client.exceptions.InvalidParameterValueException:
            print("waiting for the new role to reach Lambda")
            time.sleep(3)
    raise RuntimeError("Lambda could not use the role")


def deploy_function():
    """Creates the function from build/undercover-app.zip, or updates its code and settings in place (its address stays the same)."""
    zip_bytes = ZIP.read_bytes()
    settings = lambda_settings()
    try:
        lambda_client.get_function(FunctionName=FUNCTION_NAME)
    except lambda_client.exceptions.ResourceNotFoundException:
        create_function(zip_bytes, settings)
        lambda_client.get_waiter("function_active_v2").wait(FunctionName=FUNCTION_NAME)
        print(f"created function {FUNCTION_NAME}")
        return
    lambda_client.update_function_code(FunctionName=FUNCTION_NAME, ZipFile=zip_bytes)
    lambda_client.get_waiter("function_updated_v2").wait(FunctionName=FUNCTION_NAME)  # Lambda refuses a second change while one is under way
    lambda_client.update_function_configuration(FunctionName=FUNCTION_NAME, Role=app_role["Arn"], **settings)
    lambda_client.get_waiter("function_updated_v2").wait(FunctionName=FUNCTION_NAME)
    print(f"updated function {FUNCTION_NAME}")


deploy_function()

updated function undercover-app


## 7. The public address

A Function URL gives the function a public HTTPS address at no charge. Anyone can reach it, which is why the agent's routes need the API key and the handlers sign in. It needs two permissions, or every request gets a 403.

In [27]:
def ensure_url():
    """Gives the function a public HTTPS address, once (free). Returns it without the final slash."""
    try:
        config = lambda_client.create_function_url_config(FunctionName=FUNCTION_NAME, AuthType="NONE")
    except lambda_client.exceptions.ResourceConflictException:
        config = lambda_client.get_function_url_config(FunctionName=FUNCTION_NAME)
    public_access = [  # without both, every request gets a 403
        {"StatementId": "public-url", "Action": "lambda:InvokeFunctionUrl", "FunctionUrlAuthType": "NONE"},
        {"StatementId": "public-invoke", "Action": "lambda:InvokeFunction", "InvokedViaFunctionUrl": True},
    ]
    for statement in public_access:
        try:
            lambda_client.add_permission(FunctionName=FUNCTION_NAME, Principal="*", **statement)
        except lambda_client.exceptions.ResourceConflictException:
            pass  # already there
    return config["FunctionUrl"].rstrip("/")


app_url = ensure_url()
print(app_url)

https://example.lambda-url.us-east-1.on.aws


## 8. Test

The first request takes a few seconds while Lambda starts a copy of the app; after that, a fraction of a second. Then open the address in the browser and sign in as a handler: you see the same claims as on your laptop, since both read the same Supabase database.

The agent's service is still on your laptop, where the deployed app cannot reach it. So dispatches, evaluations and rounds from the deployed dashboard say that the agent's service is not running. A sign-off works, and pays the claim.

In [28]:
def fetch(url, headers=None):
    """GET a URL. Returns the status, the start of the body and the seconds it took, also for an error status."""
    started = time.perf_counter()
    request = urllib.request.Request(url, headers=headers or {})
    try:
        with urllib.request.urlopen(request, timeout=60) as answer:
            status, body = answer.status, answer.read(300).decode("utf-8", "replace")
    except urllib.error.HTTPError as error:
        status, body = error.code, error.read(300).decode("utf-8", "replace")
    return status, body, time.perf_counter() - started


def show_logs(limit=25):
    """The latest lines of the function's log: where an error on Lambda is explained."""
    events = logs.filter_log_events(logGroupName=LOG_GROUP, limit=limit).get("events", [])
    for event in events:
        print(event["message"].rstrip())
    if not events:
        print("no log lines yet")


def check_app(url):
    """Opens the live address: the API, the dashboard, and the API key. Returns True if all three answer as they should."""
    for attempt in range(6):  # a new address and a new version take a few seconds to be reachable
        status, body, seconds = fetch(f"{url}/api/health")
        print(f"/api/health: {status} {body.strip()[:60]} in {seconds:.1f} s" + ("  (the first request starts the app)" if attempt == 0 else ""))
        if status == 200:
            break
        time.sleep(5)
    else:
        print("the API does not answer; the function's log says why:")
        show_logs()
        return False
    status, body, seconds = fetch(url + "/")
    page_ok = status == 200 and "<html" in body.lower()
    print(f"/ (the dashboard): {status} {'looks like the dashboard' if page_ok else body.strip()[:60]} in {seconds:.1f} s")

    claim_url = f"{url}/api/claims/CLM-1901"
    without_key, _, _ = fetch(claim_url)
    with_key, _, _ = fetch(claim_url, {"X-API-Key": load_env("UNDERCOVER_API_KEY")})
    print(f"CLM-1901 without the key: {without_key} (should be 401) | with the key: {with_key} (should be 200)")
    if with_key != 200:
        show_logs()
    return page_ok and without_key == 401 and with_key == 200


check_app(app_url)

/api/health: 200 {"status":"ok"} in 0.4 s  (the first request starts the app)
/ (the dashboard): 200 looks like the dashboard in 0.4 s
CLM-1901 without the key: 401 (should be 401) | with the key: 200 (should be 200)


True

## 9. Update

To publish a change, run `redeploy()`: it builds the package again and replaces the function's code in place. The address does not change. Use `frontend_changed=True` when the dashboard changed too.

In [29]:
def redeploy(frontend_changed=False):
    """Publishes a change: the dashboard again if the frontend changed, then the package, then the function (same address). Returns True if the live app answers."""
    if frontend_changed:
        build_dashboard()
    build_package()
    deploy_function()
    return check_app(app_url)


# redeploy()                       # after a change to the backend or the data
# redeploy(frontend_changed=True)  # after a change to the frontend as well

## 10. The AgentCore Gateway

**AgentCore Gateway** is a front door between an agent and its tools. It turns the app's routes into **MCP tools** that the claim agent can call, and every caller must sign its requests with AWS credentials. **AgentCore Identity** keeps the credential the Gateway needs to reach the app: the app's API key. The Gateway fetches it and sends it with every call, so the agent never sees it and it is not in the agent's code.

```
agent  ->  Gateway (signed with AWS, rules from the Policy)  ->  app on Lambda (X-API-Key added by Identity)
```

The Gateway, the policy engine and the secret that holds the key cost a little each month: section 12 deletes them. This section needs the app from section 8 to be up.

In [30]:
import asyncio
import json
import time
from datetime import datetime, timedelta, timezone

import httpx
import requests
from botocore.auth import SigV4Auth as BotocoreSigV4Auth
from botocore.awsrequest import AWSRequest

try:
    from langchain_mcp_adapters.client import MultiServerMCPClient
    from mcp.shared.exceptions import McpError
except ImportError as error:
    raise ImportError("this section needs langchain-mcp-adapters (it brings mcp): run `%pip install langchain-mcp-adapters` in a cell and restart the kernel") from error

# The clients this section and the teardown use (the same as section 5 made: running this cell is enough to run the teardown alone)
iam = aws_session.client("iam")
logs = aws_session.client("logs")
lambda_client = aws_session.client("lambda")
agentcore = aws_session.client("bedrock-agentcore-control")
cloudwatch = aws_session.client("cloudwatch")
region = aws_session.region_name
account = aws_session.client("sts").get_caller_identity()["Account"]
API_KEY = load_env("UNDERCOVER_API_KEY")

In [31]:
def wait_for(get_resource, label, limit_s=600):
    """Waits while an AgentCore resource is being created or updated, then prints its status."""
    waited = 0
    while True:
        resource = get_resource()
        if resource["status"] not in ("CREATING", "UPDATING", "SYNCHRONIZING"):
            break
        if waited >= limit_s:
            raise TimeoutError(f"{label} is still {resource['status']} after {limit_s} seconds")
        time.sleep(3)
        waited += 3
    print(f"{label}: {resource['status']}")
    for reason in resource.get("statusReasons", []):
        print(f"  {reason}")
    return resource


# Finders. Teardown uses them too, so each lists only what is named undercover...
def find_key_providers():
    pages = agentcore.get_paginator("list_api_key_credential_providers").paginate()
    return [p["name"] for page in pages for p in page["credentialProviders"] if p["name"].startswith("undercover")]


def find_gateways():
    pages = agentcore.get_paginator("list_gateways").paginate()
    return [g for page in pages for g in page["items"] if g["name"].startswith(GATEWAY_NAME)]


def find_targets(gateway_id):
    pages = agentcore.get_paginator("list_gateway_targets").paginate(gatewayIdentifier=gateway_id)
    return [t for page in pages for t in page["items"]]


def find_policy_engines():
    pages = agentcore.get_paginator("list_policy_engines").paginate()
    return [e for page in pages for e in page["policyEngines"] if e["name"].startswith("undercover")]


def find_policies(engine_id):
    pages = agentcore.get_paginator("list_policies").paginate(policyEngineId=engine_id)
    return [p for page in pages for p in page["policies"]]

### The API key in AgentCore Identity

The Gateway sends the app's API key with every call. It keeps the key in AgentCore Identity as an **API key credential provider**, which stores it in AWS Secrets Manager. Running this cell again updates the key.

In [32]:
if KEY_PROVIDER_NAME in find_key_providers():
    agentcore.update_api_key_credential_provider(name=KEY_PROVIDER_NAME, apiKey=API_KEY)
    print(f"credential provider {KEY_PROVIDER_NAME} already exists, key updated")
else:
    agentcore.create_api_key_credential_provider(name=KEY_PROVIDER_NAME, apiKey=API_KEY, tags=PROJECT_TAG)
    print(f"created credential provider {KEY_PROVIDER_NAME}")

provider = agentcore.get_api_key_credential_provider(name=KEY_PROVIDER_NAME)
provider_arn = provider["credentialProviderArn"]
secret_arn = provider["apiKeySecretArn"]["secretArn"]

created credential provider undercover-app-key


### The Gateway's role

The Gateway runs as a role of its own. Only gateways named `undercover-gateway` in your account can use it. For now it can only fetch the API key; section 11 lets it use the Policy.

In [33]:
gateway_trust = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": account},
                "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{region}:{account}:gateway/{GATEWAY_NAME}-*"},
            },
        }
    ],
}
try:
    gateway_role = iam.create_role(
        RoleName=GATEWAY_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(gateway_trust),
        Description="The role the UnderCover gateway runs as",
        Tags=[{"Key": key, "Value": value} for key, value in PROJECT_TAG.items()],
    )["Role"]
    print(f"created role {GATEWAY_ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    gateway_role = iam.get_role(RoleName=GATEWAY_ROLE_NAME)["Role"]
    print(f"role {GATEWAY_ROLE_NAME} already exists")

# To fetch the key, the Gateway goes through its own workload identity and the token vault where Identity keeps credentials
workload_directory = f"arn:aws:bedrock-agentcore:{region}:{account}:workload-identity-directory/default"
workload_identity = [workload_directory, f"{workload_directory}/workload-identity/{GATEWAY_NAME}-*"]
token_vault = provider_arn.split("/apikeycredentialprovider/")[0]
api_key_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": "bedrock-agentcore:GetWorkloadAccessToken", "Resource": workload_identity},
        {"Effect": "Allow", "Action": "bedrock-agentcore:GetResourceApiKey", "Resource": [provider_arn, token_vault, *workload_identity]},
        {"Effect": "Allow", "Action": "secretsmanager:GetSecretValue", "Resource": secret_arn},
    ],
}
iam.put_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName="undercover-gateway-api-key", PolicyDocument=json.dumps(api_key_policy))
print("the role can fetch the app's API key")

created role undercover-gateway-role
the role can fetch the app's API key


### The Gateway

The Gateway speaks MCP, and callers sign their requests with their AWS credentials (`AWS_IAM`), so only users and roles in your account that may invoke it can use it. Creating it takes a minute or two.

In [34]:
def create_gateway():
    """Creates the Gateway, waiting for its new role to reach AgentCore."""
    for attempt in range(10):
        try:
            return agentcore.create_gateway(
                name=GATEWAY_NAME,
                description="UnderCover's systems as tools for the claim agent",
                roleArn=gateway_role["Arn"],
                protocolType="MCP",
                authorizerType="AWS_IAM",  # callers sign their requests with AWS credentials
                exceptionLevel="DEBUG",  # its error messages say what went wrong
                tags=PROJECT_TAG,
            )
        except agentcore.exceptions.ValidationException as error:
            if "role" not in str(error).lower():
                raise
            print("waiting for the new role to reach AgentCore")
            time.sleep(3)
    raise RuntimeError("AgentCore could not use the role")


existing = find_gateways()
gateway_id = existing[0]["gatewayId"] if existing else create_gateway()["gatewayId"]
gateway = wait_for(lambda: agentcore.get_gateway(gatewayIdentifier=gateway_id), f"Gateway {gateway_id}")
gateway_arn = gateway["gatewayArn"]
gateway_url = gateway["gatewayUrl"]
print(gateway_url)

Gateway undercover-gateway-0kmmoobdox: READY
https://undercover-gateway-0kmmoobdox.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp


### The agent's own API document

The app serves an OpenAPI document with only the eight operations the claim agent uses (`GATEWAY_OPERATIONS` in `backend/app/api.py`). Signing off, the dashboard's routes and the evaluations are not in it, so the agent cannot call them through the Gateway.

In [35]:
# The app serves a document with only the operations the claim agent uses (the backend's GATEWAY_OPERATIONS).
# Each route becomes a tool named by its operationId. FastAPI leaves the app's address out, so we add it.
agent_api = requests.get(f"{app_url}/api/agent/openapi.json", headers={"X-API-Key": API_KEY}, timeout=30).json()
agent_api["servers"] = [{"url": app_url}]

for path, operations in agent_api["paths"].items():
    for method, operation in operations.items():
        print(f"{method.upper():<6}{operation['operationId']:<20}{path}")

GET   get_claim           /api/claims/{claim_id}
GET   get_policy          /api/claims/{claim_id}/policy
GET   get_claim_history   /api/claims/{claim_id}/history
GET   search_documents    /api/documents/search
GET   read_section        /api/documents/{document}/sections/{section}
POST  calculate_payout    /api/payouts/calculate
POST  record_decision     /api/claims/{claim_id}/decision
POST  pay_claim           /api/claims/{claim_id}/pay


### The target

The app becomes the Gateway's **target**, with that document inline and the API key sent in the `X-API-Key` header. Running this cell again updates the target with a fresh document.

In [40]:
target_settings = {
    "gatewayIdentifier": gateway_id,
    "name": TARGET_NAME,
    "description": "The UnderCover app's routes for the claim agent",
    "targetConfiguration": {"mcp": {"openApiSchema": {"inlinePayload": json.dumps(agent_api)}}},
    "credentialProviderConfigurations": [
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {"providerArn": provider_arn, "credentialParameterName": "X-API-Key", "credentialLocation": "HEADER"}
            },
        }
    ],
}

existing = [t for t in find_targets(gateway_id) if t["name"] == TARGET_NAME]
if existing:
    target_id = existing[0]["targetId"]
    agentcore.update_gateway_target(targetId=target_id, **target_settings)
else:
    target_id = agentcore.create_gateway_target(**target_settings)["targetId"]
target = wait_for(lambda: agentcore.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id), f"Target {target_id}")

Target EICWGLN2PP: READY


### Signing our calls, and the tools

Each request to the Gateway is signed with AWS Signature Version 4, using your IAM user's credentials, as boto3 signs its own calls: `SigV4Auth` does that for the HTTP requests the MCP client sends. `call_tool` calls one tool and prints what comes back.

In [44]:
class SigV4Auth(httpx.Auth):
    """Signs each request with AWS Signature Version 4, as boto3 does for its own calls."""

    requires_request_body = True

    def __init__(self, session, service="bedrock-agentcore"):
        self.credentials = session.get_credentials()
        self.region = session.region_name
        self.service = service

    def auth_flow(self, request):
        headers = {"Host": request.url.netloc.decode(), "Content-Type": request.headers.get("content-type", "")}
        aws_request = AWSRequest(method=request.method, url=str(request.url), data=request.content, headers=headers)
        BotocoreSigV4Auth(self.credentials.get_frozen_credentials(), self.service, self.region).add_auth(aws_request)
        for name in ("Authorization", "X-Amz-Date", "X-Amz-Security-Token"):
            if name in aws_request.headers:
                request.headers[name] = aws_request.headers[name]
        yield request

In [ ]:
from langchain_core.tools import ToolException

client = MultiServerMCPClient(
    {"undercover": {"transport": "streamable_http", "url": gateway_url, "auth": SigV4Auth(aws_session)}}
)
tools = {tool.name: tool for tool in await client.get_tools()}
for tool in tools.values():
    print(f"{tool.name:<36}{tool.description}")


def tool_text(result):
    """The text of a tool's answer. Depending on the version of langchain-mcp-adapters it is a string, a list of content blocks ({"type": "text", "text": ...}) or a list of strings."""
    if isinstance(result, str):
        return result
    if isinstance(result, (list, tuple)):
        return "\n".join(tool_text(item) for item in result)
    if isinstance(result, dict):
        return str(result.get("text", result))
    return str(getattr(result, "text", result))


def leaves(error):
    """The errors inside an exception group, however deeply nested (the MCP client runs its calls in task groups, which wrap what goes wrong)."""
    if isinstance(error, BaseExceptionGroup):
        return [leaf for inner in error.exceptions for leaf in leaves(inner)]
    return [error]


def describe_error(error):
    """What to print for one error of a tool call. The Gateway's refusal and the app's error answer look alike to the library, so they are told apart by the message."""
    text = str(error)
    if isinstance(error, ToolException) and "API request failed with status" in text:
        # An error the app sent back, which the agent would read as the tool's result
        return f"Error from the app (the agent would read this): {text[:300]}"
    if isinstance(error, (ToolException, McpError)):
        return f"Refused by the Gateway: {text[:300]}"
    return f"{type(error).__name__}: {text[:300]}"


async def call_tool(operation, **arguments):
    """Calls one of the app's routes through the Gateway and prints what comes back. An error comes back as an exception, possibly inside an exception group: it is printed, not raised."""
    try:
        result = await tools[f"{TARGET_NAME}___{operation}"].ainvoke(arguments)
        print(tool_text(result)[:300])
    except (ToolException, McpError, BaseExceptionGroup) as raised:
        for error in leaves(raised):
            print(describe_error(error))


In [46]:
# A new role can take up to a minute to reach AgentCore, so the first calls may fail with "Failed to obtain execution role credentials"
for attempt in range(20):
    try:
        text = tool_text(await tools[f"{TARGET_NAME}___get_claim"].ainvoke({"claim_id": "CLM-1901"}))
    except ToolException as error:  # while the role has not arrived, the error may come as an exception
        text = str(error)
    if "execution role" not in text:
        break
    print("waiting for the Gateway's role to reach AgentCore")
    await asyncio.sleep(5)
print(text[:300])

# Two more calls: a search, and an error from the app (it comes back as the tool's result, for the agent to read)
await call_tool("search_documents", query="unattended")
await call_tool("get_claim", claim_id="CLM-9999")

{"claim":{"claim_id":"CLM-1901","policy_id":"POL-2901","customer_id":"CUS-3901","claim_type":"baggage","submitted_on":"2026-09-02","incident_date":"2026-08-25","incident_location":"Cancun, Mexico","summary":"Watch stolen from the beach while swimming","story":"On August 25 I went for a swim at the b
[{"document":"claims_manual","section":7,"title":"Interpretation notes","snippet":"n. They have been agreed with the underwriters and they bind every decision. **Unattended belongings.** Belongings are unattended when the customer could neith"},{"document":"policy_wording","section":5,"title":"Bagga
Error from the app (the agent would read this): Client error: API request failed with status: 404 - {"detail":"unknown claim CLM-9999"}


## 11. The Policy

A **policy engine** holds rules written in Cedar, and the Gateway asks it about every tool call. With an engine attached, a call is refused unless some rule permits it, so the first rule permits every tool of our target. The second **forbids `pay_claim` with an amount over $1,000**, and a forbid always wins over a permit. The amount is a JSON number, which Cedar reads as a `decimal`, so the comparison is `greaterThan` rather than `>`.

A **third rule** (the lab's challenge) forbids `record_decision` when the decision is an **approval with a payout over $5,000**: a large approval is something a handler should look at, whatever the amount claimed. Like the others it is tried in log-only mode first.

This is a second layer: the app has its own $1,000 limit behind it, and a handler's sign-off is the only way a payment over the limit is paid (the sign-off is not a tool on the Gateway).

In [55]:
existing = [e for e in find_policy_engines() if e["name"] == ENGINE_NAME]
if existing:
    engine_id = existing[0]["policyEngineId"]
else:
    engine_id = agentcore.create_policy_engine(name=ENGINE_NAME, description="Rules for the UnderCover gateway", tags=PROJECT_TAG)["policyEngineId"]
engine = wait_for(lambda: agentcore.get_policy_engine(policyEngineId=engine_id), f"Policy engine {engine_id}")
engine_arn = engine["policyEngineArn"]

# The Gateway's role needs permission to use the engine
policy_engine_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": "bedrock-agentcore:GetPolicyEngine", "Resource": engine_arn},
        {"Effect": "Allow", "Action": ["bedrock-agentcore:AuthorizeAction", "bedrock-agentcore:PartiallyAuthorizeActions"], "Resource": [engine_arn, gateway_arn]},
    ],
}
iam.put_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName="undercover-gateway-policy-engine", PolicyDocument=json.dumps(policy_engine_policy))
print("the role can use the policy engine")

Policy engine undercover_policy_engine-g5rbig3ua4: ACTIVE
the role can use the policy engine


### Attaching it in log-only mode

In `LOG_ONLY` mode the engine records what it would refuse and lets every call through. That is how you try a new rule on real traffic before enforcing it. `update_gateway` needs every setting sent again, and the role's new permission can take a few seconds to arrive.

In [56]:
def attach_policy_engine(mode):
    """Attaches the policy engine to the Gateway in LOG_ONLY mode (records what it would refuse, lets every call through) or ENFORCE mode."""
    for attempt in range(20):
        try:
            agentcore.update_gateway(
                gatewayIdentifier=gateway_id,
                name=GATEWAY_NAME,
                description=gateway["description"],
                roleArn=gateway_role["Arn"],
                protocolType="MCP",
                authorizerType="AWS_IAM",
                exceptionLevel="DEBUG",
                policyEngineConfiguration={"arn": engine_arn, "mode": mode},
            )
            break
        except agentcore.exceptions.ValidationException as error:
            if "GetPolicyEngine" not in str(error):
                raise
            print("waiting for the role's new permission to reach AgentCore")
            time.sleep(5)
    updated = wait_for(lambda: agentcore.get_gateway(gatewayIdentifier=gateway_id), f"Gateway {gateway_id}")
    print(f"policy engine mode: {updated['policyEngineConfiguration']['mode']}")


attach_policy_engine("LOG_ONLY")  # try a new rule on real traffic before enforcing it

Gateway undercover-gateway-0kmmoobdox: READY
policy engine mode: LOG_ONLY


### The rules in Cedar

In [57]:
policies = {
    "undercover_allow_agent_tools": f"""permit(
  principal is AgentCore::IamEntity,
  action in AgentCore::Action::"{TARGET_NAME}",
  resource == AgentCore::Gateway::"{gateway_arn}"
);""",
    "undercover_payment_limit": f"""forbid(
  principal is AgentCore::IamEntity,
  action == AgentCore::Action::"{TARGET_NAME}___pay_claim",
  resource == AgentCore::Gateway::"{gateway_arn}"
)
when {{
  context.input.amount.greaterThan(decimal("1000.00"))
}};""",
    # The challenge: no approval with a payout over $5,000 is recorded through the Gateway. A string is compared with ==, the payout as a decimal.
    "undercover_large_approval": f"""forbid(
  principal is AgentCore::IamEntity,
  action == AgentCore::Action::"{TARGET_NAME}___record_decision",
  resource == AgentCore::Gateway::"{gateway_arn}"
)
when {{
  context.input.decision == "approve" &&
  context.input.payout.greaterThan(decimal("5000.00"))
}};""",
}
print(policies["undercover_payment_limit"])
print(policies["undercover_large_approval"])

existing = {p["name"]: p for p in find_policies(engine_id)}
for name, statement in policies.items():
    if name in existing:
        policy_id = existing[name]["policyId"]
    else:
        policy_id = agentcore.create_policy(policyEngineId=engine_id, name=name, definition={"cedar": {"statement": statement}})["policyId"]
    wait_for(lambda: agentcore.get_policy(policyEngineId=engine_id, policyId=policy_id), f"Policy {name}")

forbid(
  principal is AgentCore::IamEntity,
  action == AgentCore::Action::"undercover-app___pay_claim",
  resource == AgentCore::Gateway::"arn:aws:bedrock-agentcore:us-east-1:123456789012:gateway/undercover-gateway-0kmmoobdox"
)
when {
  context.input.amount.greaterThan(decimal("1000.00"))
};
forbid(
  principal is AgentCore::IamEntity,
  action == AgentCore::Action::"undercover-app___record_decision",
  resource == AgentCore::Gateway::"arn:aws:bedrock-agentcore:us-east-1:123456789012:gateway/undercover-gateway-0kmmoobdox"
)
when {
  context.input.decision == "approve" &&
  context.input.payout.greaterThan(decimal("5000.00"))
};
Policy undercover_allow_agent_tools: ACTIVE
Policy undercover_payment_limit: ACTIVE
Policy undercover_large_approval: ACTIVE


### Which layer says no

Suppose the injected note on CLM-1901 had fooled the agent, and it tried to pay the $4,800. In log-only mode the Gateway lets the call through and the app's own limit refuses it. The same goes for a $6,000 approval, which the third rule would refuse.

In [58]:
# In LOG_ONLY mode the Gateway lets both calls through. The app's own limit refuses the $4,800; the $400 passes the Policy
# and the app refuses it for its own reason (CLM-1901 was never approved).
await call_tool("pay_claim", claim_id="CLM-1901", amount=4800.0)
await call_tool("pay_claim", claim_id="CLM-1901", amount=400.0)

# The third rule, the same way. CLM-9999 does not exist, so nothing is ever recorded: the app answers "unknown claim" to both calls
# (and in the next cell the Gateway itself refuses the $6,000 before the app is reached).
LARGE_APPROVAL = dict(claim_id="CLM-9999", decision="approve", payout=6000.0, calculation=[], rationale="rule test", citations=[], model="rule test")
await call_tool("record_decision", **LARGE_APPROVAL)
await call_tool("record_decision", **{**LARGE_APPROVAL, "payout": 4000.0})  # within $5,000: passes the Policy either way

Error from the app (the agent would read this): Client error: API request failed with status: 403 - {"detail":"a payment of $4,800.00 is over the approval limit of $1,000.00"}
Error from the app (the agent would read this): Client error: API request failed with status: 409 - {"detail":"CLM-1901 has no approval to pay"}
Error from the app (the agent would read this): Client error: API request failed with status: 404 - {"detail":"unknown claim CLM-9999"}
Error from the app (the agent would read this): Client error: API request failed with status: 404 - {"detail":"unknown claim CLM-9999"}


In [60]:
def leaves(error):
    """The errors inside an exception group, however deeply nested."""
    if isinstance(error, BaseExceptionGroup):
        return [leaf for inner in error.exceptions for leaf in leaves(inner)]
    return [error]


def describe_error(error):
    """What to print for one error of a tool call."""
    text = str(error)
    if isinstance(error, ToolException) and "API request failed with status" in text:
        return f"Error from the app (the agent would read this): {text[:300]}"
    if isinstance(error, (ToolException, McpError)):
        return f"Refused by the Gateway: {text[:300]}"
    return f"{type(error).__name__}: {text[:300]}"


async def call_tool(operation, **arguments):
    """Calls one of the app's routes through the Gateway and prints what comes back."""
    try:
        result = await tools[f"{TARGET_NAME}___{operation}"].ainvoke(arguments)
        print(tool_text(result)[:300])
    except (ToolException, McpError, BaseExceptionGroup) as raised:
        for error in leaves(raised):
            print(describe_error(error))


### Enforcing the Policy

Now the Gateway refuses the $4,800 itself, and the call never reaches the app. So does the $6,000 approval.

In [61]:
# Enforcing: now the Gateway itself refuses the $4,800, and the call never reaches the app. The $400 still goes through to the app.
attach_policy_engine("ENFORCE")
await call_tool("pay_claim", claim_id="CLM-1901", amount=4800.0)
await call_tool("pay_claim", claim_id="CLM-1901", amount=400.0)
await call_tool("record_decision", **LARGE_APPROVAL)  # $6,000 approval: refused by the Gateway now
await call_tool("record_decision", **{**LARGE_APPROVAL, "payout": 4000.0})  # $4,000: still reaches the app (unknown claim)

Gateway undercover-gateway-0kmmoobdox: READY
policy engine mode: ENFORCE
Refused by the Gateway: Tool Execution Denied: Tool call not allowed due to policy enforcement [Policy evaluation denied due to undercover_payment_limit-9su4iasqe5]
Error from the app (the agent would read this): Client error: API request failed with status: 409 - {"detail":"CLM-1901 has no approval to pay"}
Refused by the Gateway: Tool Execution Denied: Tool call not allowed due to policy enforcement [Policy evaluation denied due to undercover_large_approval-qcwr3f0z1h]
Error from the app (the agent would read this): Client error: API request failed with status: 404 - {"detail":"unknown claim CLM-9999"}


### The engine's decisions

The policy engine counts its decisions in CloudWatch, a minute or two behind, so run this cell again in a couple of minutes if it shows zeros.

In [62]:
def decisions(metric, mode):
    """The engine's decisions of one kind in one mode over the last three hours (CloudWatch is a minute or two behind)."""
    now = datetime.now(timezone.utc)
    stats = cloudwatch.get_metric_statistics(
        Namespace="AWS/Bedrock-AgentCore",
        MetricName=metric,
        Dimensions=[
            {"Name": "OperationName", "Value": "AuthorizeAction"},
            {"Name": "Mode", "Value": mode},
            {"Name": "PolicyEngine", "Value": engine_id},
        ],
        StartTime=now - timedelta(hours=3),
        EndTime=now,
        Period=3600,
        Statistics=["Sum"],
    )
    return int(sum(point["Sum"] for point in stats["Datapoints"]))


for mode in ["LOG_ONLY", "ENFORCE"]:
    print(f"{mode:<10}allowed {decisions('AllowDecisions', mode)}, refused {decisions('DenyDecisions', mode)}")

LOG_ONLY  allowed 4, refused 4
ENFORCE   allowed 0, refused 4


## 12. Teardown

`teardown()` lists what this notebook made and what of it still exists, and deletes nothing. `teardown(confirm=True)` deletes the **AgentCore pieces** (targets, then their Gateway; policies, then their engine; the key provider, which also removes the secret that held the key; the Gateway's role): they bill by the month. The app on Lambda costs nothing while nobody uses it, so it stays unless you also pass `app=True`, which deletes the function (and its address), its role and its logs.

To run it by itself (after a restart), run the names cell in section 1, the AWS connection cell at the top, and the two cells that start section 10 (the clients and the finders).

In [63]:
def find_roles():
    pages = iam.get_paginator("list_roles").paginate()
    return [r["RoleName"] for page in pages for r in page["Roles"] if r["RoleName"].startswith("undercover")]


def show_resources():
    """Prints what this notebook made that still exists."""
    function_pages = lambda_client.get_paginator("list_functions").paginate()
    found = {
        "Lambda functions": [f["FunctionName"] for page in function_pages for f in page["Functions"] if f["FunctionName"].startswith("undercover")],
        "Gateways": [g["name"] for g in find_gateways()],
        "Policy engines": [e["name"] for e in find_policy_engines()],
        "API key providers": find_key_providers(),
        "IAM roles": find_roles(),
    }
    for label, names in found.items():
        print(f"{label + ':':<20}{', '.join(names) or 'none'}")


def delete_role(name):
    """Deletes a role and its inline policies (a role cannot be deleted while it has any)."""
    for policy_name in iam.list_role_policies(RoleName=name)["PolicyNames"]:
        iam.delete_role_policy(RoleName=name, PolicyName=policy_name)
    iam.delete_role(RoleName=name)
    print(f"deleted role {name}")


def teardown(confirm=False, app=False):
    """Lists what exists; with confirm=True, deletes the AgentCore pieces, and with app=True also the Lambda function, its role and its logs."""
    show_resources()
    if not confirm:
        print("\nnothing deleted. To delete the AgentCore pieces: teardown(confirm=True). To delete the app too: teardown(confirm=True, app=True)")
        return

    for summary in find_gateways():  # targets go before their gateway
        for target_summary in find_targets(summary["gatewayId"]):
            agentcore.delete_gateway_target(gatewayIdentifier=summary["gatewayId"], targetId=target_summary["targetId"])
            print(f"deleted target {target_summary['name']}")
        while find_targets(summary["gatewayId"]):
            time.sleep(3)
        agentcore.delete_gateway(gatewayIdentifier=summary["gatewayId"])
        print(f"deleted gateway {summary['name']}")

    for summary in find_policy_engines():  # policies go before their engine
        for policy_summary in find_policies(summary["policyEngineId"]):
            agentcore.delete_policy(policyEngineId=summary["policyEngineId"], policyId=policy_summary["policyId"])
            print(f"deleted policy {policy_summary['name']}")
        while find_policies(summary["policyEngineId"]):
            time.sleep(3)
        agentcore.delete_policy_engine(policyEngineId=summary["policyEngineId"])
        print(f"deleted policy engine {summary['name']}")

    for name in find_key_providers():  # AgentCore deletes the secret that held the key a few seconds later
        agentcore.delete_api_key_credential_provider(name=name)
        print(f"deleted credential provider {name}")

    if GATEWAY_ROLE_NAME in find_roles():
        delete_role(GATEWAY_ROLE_NAME)

    if app:
        try:
            lambda_client.delete_function(FunctionName=FUNCTION_NAME)  # its address goes with it
            print(f"deleted function {FUNCTION_NAME}")
        except lambda_client.exceptions.ResourceNotFoundException:
            print(f"function {FUNCTION_NAME} was not there")
        if APP_ROLE_NAME in find_roles():
            delete_role(APP_ROLE_NAME)
        try:
            logs.delete_log_group(logGroupName=LOG_GROUP)
            print(f"deleted log group {LOG_GROUP}")
        except logs.exceptions.ResourceNotFoundException:
            print(f"log group {LOG_GROUP} was not there")

    time.sleep(10)
    print()
    show_resources()


teardown()  # only lists

Lambda functions:   undercover-app
Gateways:           undercover-gateway
Policy engines:     undercover_policy_engine
API key providers:  undercover-app-key
IAM roles:          undercover-app-role, undercover-gateway-role

nothing deleted. To delete the AgentCore pieces: teardown(confirm=True). To delete the app too: teardown(confirm=True, app=True)
